# RQ6: Robustness and Generalization
**Research Question:** How robust is the best supervised learning model under different train-test splits, cross-validation settings, and data perturbation scenarios?

**Model:** XGBoost (best from RQ2)  
**Scenarios:** Standard split, 5-fold CV, 10-fold CV, 10% noise, 20% missingness

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os

os.makedirs('figures', exist_ok=True)
os.makedirs('tables', exist_ok=True)

In [2]:
DATA_PATH = '/kaggle/input/datasets/karansridhar/karan-ue-ml-assignment-dataset/marketing_and_product_performance.csv'
df = pd.read_csv(DATA_PATH)
TARGET = 'Revenue_Generated'

id_cols = [c for c in df.columns if 'ID' in c or 'id' in c.lower()]
df = df.drop(columns=id_cols, errors='ignore').dropna(subset=[TARGET])
cat_cols = df.select_dtypes(include='object').columns
le = LabelEncoder()
for c in cat_cols:
    df[c] = le.fit_transform(df[c].fillna('Unknown').astype(str))

X_full = df.drop(columns=[TARGET])
y_full = df[TARGET]
imputer = SimpleImputer(strategy='median')
X_clean = pd.DataFrame(imputer.fit_transform(X_full), columns=X_full.columns)
y_clean = y_full.reset_index(drop=True)

model_base = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0)

In [3]:
results = []

# --- Scenario 1: Standard 80/20 split ---
Xtr, Xte, ytr, yte = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)
model_base.fit(Xtr, ytr)
p = model_base.predict(Xte)
results.append({'Scenario': 'Standard Split (80/20)',
                'MAE': round(mean_absolute_error(yte,p),4),
                'RMSE': round(np.sqrt(mean_squared_error(yte,p)),4),
                'R²': round(r2_score(yte,p),4), 'Std Dev': '-'})

# --- Scenario 2: 5-fold CV ---
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)
cv5 = cross_val_score(XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
                      X_clean, y_clean, cv=kf5, scoring='r2')
cv5_mae = cross_val_score(XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
                          X_clean, y_clean, cv=kf5, scoring='neg_mean_absolute_error')
results.append({'Scenario': '5-Fold CV',
                'MAE': round(-cv5_mae.mean(),4),
                'RMSE': '-',
                'R²': round(cv5.mean(),4),
                'Std Dev': f'±{cv5.std():.4f}'})

# --- Scenario 3: 10-fold CV ---
kf10 = KFold(n_splits=10, shuffle=True, random_state=42)
cv10 = cross_val_score(XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
                       X_clean, y_clean, cv=kf10, scoring='r2')
cv10_mae = cross_val_score(XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
                           X_clean, y_clean, cv=kf10, scoring='neg_mean_absolute_error')
results.append({'Scenario': '10-Fold CV',
                'MAE': round(-cv10_mae.mean(),4),
                'RMSE': '-',
                'R²': round(cv10.mean(),4),
                'Std Dev': f'±{cv10.std():.4f}'})

# --- Scenario 4: 10% Gaussian noise ---
np.random.seed(42)
X_noisy = X_clean.copy()
noise_cols = X_noisy.select_dtypes(include=np.number).columns
X_noisy[noise_cols] += np.random.normal(0, X_noisy[noise_cols].std() * 0.10)
Xtr4, Xte4, ytr4, yte4 = train_test_split(X_noisy, y_clean, test_size=0.2, random_state=42)
m4 = XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
m4.fit(Xtr4, ytr4)
p4 = m4.predict(Xte4)
results.append({'Scenario': '10% Noise Added',
                'MAE': round(mean_absolute_error(yte4,p4),4),
                'RMSE': round(np.sqrt(mean_squared_error(yte4,p4)),4),
                'R²': round(r2_score(yte4,p4),4), 'Std Dev': '-'})

# --- Scenario 5: 20% missingness ---
X_missing = X_clean.copy()
mask = np.random.rand(*X_missing.shape) < 0.20
X_missing[mask] = np.nan
X_miss_imp = pd.DataFrame(imputer.fit_transform(X_missing), columns=X_missing.columns)
Xtr5, Xte5, ytr5, yte5 = train_test_split(X_miss_imp, y_clean, test_size=0.2, random_state=42)
m5 = XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
m5.fit(Xtr5, ytr5)
p5 = m5.predict(Xte5)
results.append({'Scenario': '20% Missingness',
                'MAE': round(mean_absolute_error(yte5,p5),4),
                'RMSE': round(np.sqrt(mean_squared_error(yte5,p5)),4),
                'R²': round(r2_score(yte5,p5),4), 'Std Dev': '-'})

df_results = pd.DataFrame(results)[['Scenario','MAE','RMSE','R²','Std Dev']]
df_results.to_csv('tables/RQ6_Table6_Robustness.csv', index=False)
print(df_results)

                 Scenario         MAE        RMSE      R²  Std Dev
0  Standard Split (80/20)  25223.6129  29267.4599 -0.0505        -
1               5-Fold CV  26039.5282           - -0.1553  ±0.0160
2              10-Fold CV  26055.2011           - -0.1540  ±0.0182
3         10% Noise Added  26014.9583  30652.8088 -0.1523        -
4         20% Missingness  25860.3925  30652.7874 -0.1523        -


In [4]:
# Figure 6 — R² across scenarios (line chart)
r2_vals  = [float(r) if r != '-' else np.nan for r in df_results['R²']]
mae_vals = [float(m) if m != '-' else np.nan for m in df_results['MAE']]
scenarios = df_results['Scenario'].tolist()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

ax1.plot(scenarios, r2_vals, marker='o', color='#1E88E5', linewidth=2.5, markersize=9)
ax1.fill_between(scenarios, r2_vals, alpha=0.1, color='#1E88E5')
ax1.set_ylabel('R²', fontsize=12)
ax1.set_title('Figure 6: Robustness of XGBoost Under Different Experimental Conditions\n(Marketing and Product Performance Dataset)', fontsize=13, fontweight='bold')
ax1.grid(alpha=0.3)
ax1.set_ylim(0, 1)

ax2.plot(scenarios, mae_vals, marker='s', color='#E53935', linewidth=2.5, markersize=9)
ax2.fill_between(scenarios, mae_vals, alpha=0.1, color='#E53935')
ax2.set_ylabel('MAE', fontsize=12)
ax2.set_xlabel('Scenario', fontsize=12)
ax2.grid(alpha=0.3)
plt.xticks(rotation=15, ha='right', fontsize=10)
plt.tight_layout()
plt.savefig('figures/RQ6_Figure6_Robustness.pdf', bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.
